# Knowledge Vault Agent — Demo

This notebook demonstrates calling the Knowledge Vault Cortex Agent via REST API,
including org-based disambiguation of shared abbreviations (e.g., "CR").

Uses a connection from `~/.snowflake/config.toml` and the
thread API for multi-turn context management.

In [ ]:
import json
import tomllib
from pathlib import Path

import requests

# Load connection config from ~/.snowflake/config.toml
config_path = Path.home() / ".snowflake" / "config.toml"
with open(config_path, "rb") as f:
    config = tomllib.load(f)

# Update this to match your connection name in config.toml
CONNECTION_NAME = "parker_demo"
conn = config["connections"][CONNECTION_NAME]
ACCOUNT = conn["account"]
PAT = conn["password"]  # PAT token stored as password

# Derive the account URL (underscores become hyphens in the hostname)
ACCOUNT_URL = f"https://{ACCOUNT.replace('_', '-').lower()}.snowflakecomputing.com"

# Agent location
AGENT_DB = "DOC_KNOWLEDGE_GRAPH_DEMO"
AGENT_SCHEMA = "SERVING"
AGENT_NAME = "KNOWLEDGE_VAULT_AGENT"

AGENT_ENDPOINT = f"{ACCOUNT_URL}/api/v2/databases/{AGENT_DB}/schemas/{AGENT_SCHEMA}/agents/{AGENT_NAME}:run"
THREAD_ENDPOINT = f"{ACCOUNT_URL}/api/v2/cortex/threads"

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Authorization": f"Bearer {PAT}",
}

print(f"Account: {ACCOUNT}")
print(f"Agent:   {AGENT_DB}.{AGENT_SCHEMA}.{AGENT_NAME}")
print(f"Endpoint: {AGENT_ENDPOINT}")

KeyError: 'default'

In [ ]:
def create_thread() -> str:
    """Create a new conversation thread. Returns the thread_id."""
    resp = requests.post(
        THREAD_ENDPOINT,
        headers=HEADERS,
        json={"origin_application": "kv_demo"}
    )
    resp.raise_for_status()
    thread_data = resp.json()
    return thread_data["thread_id"]


def call_agent(message: str, thread_id: str, parent_message_id: str = "0", system_message: str = None) -> tuple[str, str]:
    """Send a message to the agent within a thread.
    
    Args:
        message: The user's question
        thread_id: The conversation thread ID
        parent_message_id: ID of the previous message in the thread (\"0\" for first message)
        system_message: Optional system prompt (e.g., user's team for disambiguation)
    
    Returns:
        Tuple of (response_text, message_id) — use message_id as parent_message_id for next turn
    """
    messages = []
    if system_message:
        messages.append({"role": "system", "content": [{"type": "text", "text": system_message}]})
    messages.append({"role": "user", "content": [{"type": "text", "text": message}]})
    
    payload = {
        "model": f"{AGENT_DB}.{AGENT_SCHEMA}.{AGENT_NAME}",
        "thread_id": thread_id,
        "parent_message_id": parent_message_id,
        "messages": messages,
    }
    
    resp = requests.post(AGENT_ENDPOINT, headers=HEADERS, json=payload)
    resp.raise_for_status()
    
    # Parse SSE response
    full_text = ""
    message_id = parent_message_id
    current_event = ""
    
    for line in resp.text.strip().split("\n"):
        if line.startswith("event:"):
            current_event = line[6:].strip()
        elif line.startswith("data:"):
            try:
                data = json.loads(line[5:])
                if current_event == "response.text.delta" and "text" in data:
                    full_text += data["text"]
                if current_event == "metadata":
                    meta = data.get("metadata", data)
                    if meta.get("role") == "assistant" and "message_id" in meta:
                        message_id = str(meta["message_id"])
            except json.JSONDecodeError:
                pass
    
    return full_text, message_id

print("Helper functions ready.")

---
## Example Conversation Thread: Activation Rate Deep-Dive

This shows a real multi-turn conversation with the agent, starting from a basic
concept lookup and drilling into relationships and data sources.

In [ ]:
# Create a new thread for this conversation
thread_id = create_thread()
print(f"Thread: {thread_id}")

In [ ]:
# Turn 1: Basic concept lookup
print("USER: What is activation rate?")
print("=" * 60)
response, msg_id = call_agent(
    "What is activation rate?",
    thread_id,
    system_message="The user is on the Analytics team."
)
print(response)

In [ ]:
# Turn 2: Follow up — ask for the full concept note
print("USER: Can you pull the full concept note for Activation Rate?")
print("=" * 60)
response, msg_id = call_agent(
    "Can you pull the full concept note for Activation Rate?",
    thread_id,
    parent_message_id=msg_id
)
print(response)

In [ ]:
# Turn 3: Follow up — ask about relationships to data sources and business rules
print("USER: How is Activation Rate related to its underlying data sources and business rules?")
print("=" * 60)
response, msg_id = call_agent(
    "How is Activation Rate related to its underlying data sources and business rules?",
    thread_id,
    parent_message_id=msg_id
)
print(response)

---
## Disambiguation Examples

These examples show how the same abbreviation resolves differently based on
the user's team context provided in the system prompt.

### Analytics Team Asks About "CR"

The system prompt tells the agent the user is on the Analytics team.
"CR" should resolve to **Completion Rate**.

In [ ]:
analytics_thread = create_thread()
analytics_response, analytics_msg_id = call_agent(
    "What is CR and how do we measure it?",
    analytics_thread,
    system_message="The user is on the Analytics team."
)
print("ANALYTICS TEAM:")
print(analytics_response)

### Finance Team Asks About "CR"

Same question, different org context. "CR" should resolve to **Credit Rating**.

In [ ]:
finance_thread = create_thread()
finance_response, finance_msg_id = call_agent(
    "What is CR and how do we measure it?",
    finance_thread,
    system_message="The user is on the Finance team."
)
print("FINANCE TEAM:")
print(finance_response)

### User Corrects the Disambiguation

Continue the Analytics thread — the user says they actually meant Credit Rating.

In [ ]:
# Continue the analytics_thread from above
correction_response, correction_msg_id = call_agent(
    "Actually, I meant Credit Rating. Can you explain that instead?",
    analytics_thread,
    parent_message_id=analytics_msg_id
)
print("CORRECTION:")
print(correction_response)

---
## Other Examples

### Relationship Traversal

In [ ]:
t = create_thread()
response, _ = call_agent("What metrics are related to Cohort and how?", t)
print(response)

### Cross-Concept Reasoning

In [ ]:
t = create_thread()
response, _ = call_agent("How does plan mix affect our blended metrics?", t)
print(response)

### Entity Search

In [ ]:
t = create_thread()
response, _ = call_agent("What data sources feed into our metrics?", t)
print(response)